# mcp

> `paar-mcp`: a stdio MCP server bridging an agent (Claude Code, Codex, ...) to a live paar
> session's restricted `/agent/api` surface. The agent gets read access to every variable plus a
> persistent overlay to create its own — enforcement lives server-side in paar, this façade is the
> ergonomic tool wrapper. Requires the `mcp` extra: `pip install paar[mcp]`.

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import argparse, json
import httpx
import paar.registry as R

INSTRUCTIONS = (
    "You are connected to a live, paar-inspected Python session owned by the user. "
    "You can read every variable and create your own -- new names persist in your overlay layer "
    "across execute calls -- but you cannot mutate, rebind in place, or delete the owner's "
    "variables, and destructive file/shell escapes are blocked. A refused cell returns an error "
    "starting with 'blocked:'; rewrite it to bind results to new names instead of changing owner "
    "state (e.g. df2 = df.drop(...) rather than df.drop(..., inplace=True)). "
    "Start with list_vars to see the session, expand/grid to drill into structures cheaply, "
    "and execute for everything else. Everything you attempt is transcribed to a session "
    "notebook the owner can review (its path is in session_info).")

def _base(url=None, env=None):
    "Resolve the target paar server: explicit url > env name in the local registry > the only/first live env."
    if url: return url.rstrip('/')
    envs = R.active()
    if env:
        for e in envs:
            if e['name'] == env: return e['base']
        raise RuntimeError(f"no live paar env named {env!r}; live: {[e['name'] for e in envs]}")
    if not envs: raise RuntimeError('no live paar server found; start one with paar.fasthtml.serve() or inspector()')
    return envs[0]['base']

class PaarClient:
    "Thin HTTP client for a paar server's /agent/api surface; the target resolves per call, so late starts and restarts of the paar server are fine."
    def __init__(self, url=None, env=None): self.url, self.env = url, env
    def _req(self, method, path, params):
        try: base = _base(self.url, self.env)
        except RuntimeError as e: return json.dumps({'error': str(e)})
        r = httpx.request(method, base + path, timeout=60.0,
                          **({'data': params} if method == 'POST' else {'params': params}))
        if r.status_code >= 400 and not r.headers.get('content-type', '').startswith('application/json'):
            return json.dumps({'error': f'HTTP {r.status_code}', 'body': r.text[:500]})
        return r.text
    def get(self, path, **params): return self._req('GET', path, params)
    def post(self, path, **params): return self._req('POST', path, params)

In [ ]:
#| export
def build_mcp(url=None, env=None, name='paar'):
    "Build the FastMCP server exposing a paar session's restricted agent surface as tools."
    from mcp.server.fastmcp import FastMCP
    c = PaarClient(url, env)
    m = FastMCP(name, instructions=INSTRUCTIONS)

    @m.tool(structured_output=False)
    def list_vars(profile: str = 'standard') -> str:
        "Variables in the user's live Python session, merged with your own overlay names ('agent_names' lists yours). profile: 'minimal' | 'standard' | 'full'. Every node carries an 'accessor' list to pass to expand/grid -- prefer those tools over execute for looking around."
        return c.get('/agent/api/rows', profile=profile)

    @m.tool(structured_output=False)
    def expand(accessor: str, offset: int = 0) -> str:
        '''One level of children of the value at accessor -- a JSON list from list_vars/expand, e.g. '["data", 2]'. When a trailing load-more node appears, page with its more_offset as offset.'''
        return c.get('/agent/api/expand', accessor=accessor, offset=offset)

    @m.tool(structured_output=False)
    def grid(accessor: str, roff: int = 0, coff: int = 0) -> str:
        "A paged table window of the DataFrame/ndarray at accessor (a JSON list); roff/coff scroll rows/columns. Far cheaper than printing the object with execute."
        return c.get('/agent/api/grid', accessor=accessor, roff=roff, coff=coff)

    @m.tool(structured_output=False)
    def execute(code: str, scope: str = 'overlay') -> str:
        "Run Python against the live session under the owner's restriction list. scope='overlay' persists the names you create across calls; 'isolated' discards all writes. You can read owner variables freely; an error starting with 'blocked:' means the cell would have changed owner state -- rewrite it to bind results to NEW names (copy first if you need to mutate)."
        return c.post('/agent/api/exec', code=code, scope=scope)

    @m.tool(structured_output=False)
    def session_info() -> str:
        "The connected session's env label, access mode ('restricted' or 'readonly'), policy text, the path of the notebook transcribing your runs, and the overlay names you have created so far."
        return c.get('/agent/api/info')

    return m

def main(argv=None):
    "Console entry point: `paar-mcp [--env NAME | --url URL] [--name paar]` -- stdio MCP server for agent clients."
    p = argparse.ArgumentParser(prog='paar-mcp',
        description="stdio MCP server bridging an agent to a live paar session's restricted /agent/api surface")
    p.add_argument('--url', default=None, help='paar server base URL (default: discover via the local registry)')
    p.add_argument('--env', default=None, help='pick a registry env by name when several paar servers are live')
    p.add_argument('--name', default='paar', help='MCP server name shown to the client')
    a = p.parse_args(argv)
    build_mcp(a.url, a.env, a.name).run()

Register it with an agent client against a specific environment, e.g. for Claude Code:

```sh
claude mcp add my-session -- paar-mcp --env myrepo
```

or in Codex's `config.toml`:

```toml
[mcp_servers.my-session]
command = "paar-mcp"
args = ["--env", "myrepo"]
```

## Tests

Resolution order (explicit `--url` beats `--env` beats the only live entry) and the tool
roster the façade promises to agents.

In [ ]:
# target resolution: url > env-by-name > only/first live env
import os, tempfile
os.environ['PAAR_REG_DIR'] = tempfile.mkdtemp()
import importlib, paar.registry
importlib.reload(paar.registry)

def test_base_resolution():
    assert _base(url='http://h:1/') == 'http://h:1'
    R.register('alpha', 7001); R.register('beta', 7002)
    try:
        assert _base(env='beta') == 'http://127.0.0.1:7002'
        assert _base() in ('http://127.0.0.1:7001', 'http://127.0.0.1:7002')
        try: _base(env='nope'); assert False
        except RuntimeError as e: assert 'nope' in str(e)
    finally:
        R.unregister(7001); R.unregister(7002)
    try: _base(); assert False
    except RuntimeError as e: assert 'no live paar server' in str(e)
test_base_resolution()

In [ ]:
# the façade exposes exactly the agent-surface tools
import asyncio
m = build_mcp(url='http://127.0.0.1:9')
tools = asyncio.run(m.list_tools())
assert {t.name for t in tools} == {'list_vars', 'expand', 'grid', 'execute', 'session_info'}
assert all(t.description for t in tools)
assert 'blocked' in next(t for t in tools if t.name == 'execute').description

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()